# Chapter 3: Classification — Predicting Categories

Classification answers: **"Which one?"** or **"Yes or no?"**

- Is this email spam or not? (binary — 2 classes)
- Is this flower a setosa, versicolor, or virginica? (multi-class — 3+ classes)
- Does this patient have disease X? (binary)

The model learns a **decision boundary** that separates categories.

---
## Logistic Regression: Classification with a Curve

Despite the name, logistic regression is for **classification**, not regression.

It uses the **sigmoid function** (same one from the backpropagation chapter!) to squish any number into a probability between 0 and 1.

```
Linear:   score = w1×x1 + w2×x2 + b     → any number
Sigmoid:  probability = sigmoid(score)    → between 0 and 1
Decision: if probability > 0.5 → class 1, else class 0
```

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Example: study hours → pass/fail
hours = np.array([1, 1.5, 2, 2.5, 3, 3.5, 4, 4.5, 5, 5.5, 6, 6.5, 7, 8, 9, 10]).reshape(-1, 1)
passed = np.array([0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1])

model = LogisticRegression()
model.fit(hours, passed)

print("Study Hours → Pass/Fail Prediction")
print(f"{'Hours':>6} {'Actual':>8} {'Prob':>8} {'Predicted':>10}")
print(f"{'─'*6} {'─'*8} {'─'*8} {'─'*10}")

for h, actual in zip(hours.flatten(), passed):
    prob = model.predict_proba([[h]])[0][1]
    pred = model.predict([[h]])[0]
    label_actual = "pass" if actual else "fail"
    label_pred = "pass" if pred else "fail"
    bar = "█" * int(prob * 20) + "░" * (20 - int(prob * 20))
    print(f"{h:>6.1f} {label_actual:>8} {prob:>8.1%} {label_pred:>10}  {bar}")

# Find the decision boundary
boundary = -model.intercept_[0] / model.coef_[0][0]
print(f"\nDecision boundary: {boundary:.1f} hours")
print(f"Study > {boundary:.1f}h → model predicts PASS")
print(f"Study < {boundary:.1f}h → model predicts FAIL")

---
## Multi-class Classification

More than 2 categories. The model outputs a probability for EACH class.

In [ ]:
from sklearn.datasets import load_iris

iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.3, random_state=42
)

model = LogisticRegression(max_iter=200, random_state=42)
model.fit(X_train, y_train)

print("Iris Flower Classification (3 classes)")
print(f"Accuracy: {model.score(X_test, y_test):.1%}\n")

# Show predictions with probabilities
print(f"{'#':>3} {'Actual':>12} {'Predicted':>12} {'setosa':>8} {'versic.':>8} {'virgin.':>8} {'':>3}")
print(f"{'─'*3} {'─'*12} {'─'*12} {'─'*8} {'─'*8} {'─'*8} {'─'*3}")

probs = model.predict_proba(X_test)
preds = model.predict(X_test)
for i in range(10):
    actual = iris.target_names[y_test[i]]
    predicted = iris.target_names[preds[i]]
    match = "✓" if actual == predicted else "✗"
    print(f"{i+1:>3} {actual:>12} {predicted:>12} {probs[i][0]:>8.1%} {probs[i][1]:>8.1%} {probs[i][2]:>8.1%} {match:>3}")

print("\nThe model outputs a probability for each class.")
print("It picks the class with the highest probability.")

---
## K-Nearest Neighbors (KNN): Vote by Proximity

The simplest idea: **look at the closest neighbors and vote.**

To classify a new point:
1. Find the K closest training examples
2. Count which class appears most
3. That's your prediction

No training needed — it just memorizes all data and searches at prediction time.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

# Simple 2D example
# Two types of restaurants: fast-food vs fine-dining
# Features: [price_avg, wait_time_min]
restaurants = np.array([
    [8, 3], [10, 5], [7, 2], [12, 4], [9, 3],    # fast food
    [45, 30], [60, 45], [50, 35], [55, 40], [65, 50],  # fine dining
])
types = ["fast", "fast", "fast", "fast", "fast",
         "fine", "fine", "fine", "fine", "fine"]

knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(restaurants, types)

# Test new restaurants
print("KNN Classification: Restaurant Type")
print(f"{'Price':>8} {'Wait':>6} {'Prediction':>12} {'Neighbors':>20}")
print(f"{'─'*8} {'─'*6} {'─'*12} {'─'*20}")

test_restaurants = [[15, 8], [40, 25], [25, 15], [5, 1]]
for r in test_restaurants:
    pred = knn.predict([r])[0]
    distances, indices = knn.kneighbors([r])
    neighbor_types = [types[i] for i in indices[0]]
    print(f"${r[0]:>6} {r[1]:>5}m {pred:>12} {str(neighbor_types):>20}")

print("\n$25, 15min wait: it looks at 3 closest → votes decide.")

---
## Support Vector Machine (SVM): Find the Best Boundary

SVM finds the line (or hyperplane) that **maximizes the margin** between classes.

Think of it as: not just ANY line that separates the groups, but the line with the **widest gap** between them.

In [ ]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# Compare classifiers on iris data
classifiers = {
    "Logistic Regression": LogisticRegression(max_iter=200, random_state=42),
    "KNN (k=3)": KNeighborsClassifier(n_neighbors=3),
    "KNN (k=7)": KNeighborsClassifier(n_neighbors=7),
    "SVM (linear)": make_pipeline(StandardScaler(), SVC(kernel="linear", random_state=42)),
    "SVM (rbf)": make_pipeline(StandardScaler(), SVC(kernel="rbf", random_state=42)),
}

print("Classifier Comparison on Iris Dataset")
print(f"{'Model':<25} {'Train Acc':>10} {'Test Acc':>10}")
print(f"{'─'*25} {'─'*10} {'─'*10}")

for name, clf in classifiers.items():
    clf.fit(X_train, y_train)
    train_acc = clf.score(X_train, y_train)
    test_acc = clf.score(X_test, y_test)
    bar = "█" * int(test_acc * 20)
    print(f"{name:<25} {train_acc:>10.1%} {test_acc:>10.1%}  {bar}")

print("\nDifferent algorithms, different strengths.")
print("No single best algorithm — it depends on the data.")

---
## Confusion Matrix: Understanding Errors

Accuracy alone can be misleading. A confusion matrix shows exactly WHERE the model fails.

For binary classification:
```
                    Predicted
                  Pos     Neg
Actual  Pos    [ TP  |  FN ]
        Neg    [ FP  |  TN ]
```
- **TP** (True Positive): correctly predicted positive
- **FP** (False Positive): incorrectly predicted positive (false alarm)
- **FN** (False Negative): missed a positive (dangerous in medical tests)
- **TN** (True Negative): correctly predicted negative

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

# Use the best model from above
best_model = LogisticRegression(max_iter=200, random_state=42)
best_model.fit(X_train, y_train)
preds = best_model.predict(X_test)

cm = confusion_matrix(y_test, preds)
names = iris.target_names

print("Confusion Matrix:")
print(f"{'':>14} {'Predicted →':>30}")
print(f"{'':>14} {names[0]:>10} {names[1]:>10} {names[2]:>10}")
print(f"{'Actual ↓':>14} {'─'*10} {'─'*10} {'─'*10}")
for i, name in enumerate(names):
    row = ""
    for j in range(3):
        val = cm[i][j]
        marker = f"  {val}" if i == j else f"  {val}" if val == 0 else f" *{val}"
        row += f"{marker:>10}"
    print(f"{name:>14}{row}")

print("\n(diagonal = correct, off-diagonal = errors)")
print(f"\nDetailed Report:")
print(classification_report(y_test, preds, target_names=names))

---
## Precision vs Recall: Which Errors Matter More?

- **Precision**: Of all positive predictions, how many were correct?
  - "When the model says YES, can I trust it?"
  - Important for: spam filter (don't lose important emails)

- **Recall**: Of all actual positives, how many did we find?
  - "Does the model catch all the real cases?"
  - Important for: cancer screening (don't miss any patients)

```
Precision = TP / (TP + FP)    → accuracy of positive predictions
Recall    = TP / (TP + FN)    → coverage of actual positives
```

In [ ]:
# Precision vs Recall tradeoff example
print("When does Precision vs Recall matter?\n")
print("Scenario 1: SPAM FILTER")
print("  High precision needed: don't mark real email as spam!")
print("  Missing some spam (low recall) is OK")
print("  → False Positive is costly")
print()
print("Scenario 2: CANCER SCREENING")
print("  High recall needed: don't miss any cancer patients!")
print("  Some false alarms (low precision) are OK")
print("  → False Negative is costly")
print()
print("Scenario 3: SEARCH ENGINE")
print("  Balance both: show relevant results (precision)")
print("  AND find all relevant pages (recall)")
print("  → F1 score balances both")
print()
print("F1 = 2 × (precision × recall) / (precision + recall)")
print("F1 is high only when BOTH precision and recall are high.")

---
## Summary

| Concept | Key Point |
|---------|----------|
| **Classification** | Predict which category |
| **Logistic Regression** | Sigmoid → probability → threshold |
| **KNN** | Vote of K nearest neighbors |
| **SVM** | Maximum margin boundary |
| **Confusion Matrix** | See exactly where errors happen |
| **Precision** | "When it says yes, is it right?" |
| **Recall** | "Does it catch all the real cases?" |
| **F1 Score** | Balance of precision and recall |

**Next chapter**: Trees & Forests — decision-based models